# PumpRisk Score - Final Combined Score, Risk Tier & Backtesting

Combines:
- Stage 1 (407 stocks): Price anomaly + Volume anomaly, from `cache/stage1_leaderboard.csv`
- Stage 2 (63 shortlisted stocks): Broker dominance + Foreign flow + Fundamental divergence, from `cache/stage2/*`

Then validates the score against IDX stock suspension history (1 API call, low cost).

In [1]:
import os
import glob
import numpy as np
import pandas as pd

CACHE_DIR = "cache"
STAGE2_DIR = os.path.join(CACHE_DIR, "stage2")

leaderboard = pd.read_csv(os.path.join(CACHE_DIR, "stage1_leaderboard.csv"))
print(f"Stage 1 leaderboard: {len(leaderboard)} stocks")
leaderboard[["symbol", "stage1_score"]].head()

Stage 1 leaderboard: 407 stocks


,symbol,stage1_score
0,JECC,1.000000
1,MGNA,0.996657
2,MSKY,0.950188
3,VINS,0.857666
4,KLIN,0.821025


In [2]:
def load_stage2_folder(name: str) -> pd.DataFrame:
    files = glob.glob(os.path.join(STAGE2_DIR, name, "*.csv"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

broker_df = load_stage2_folder("broker")
foreign_df = load_stage2_folder("foreign")
news_df = load_stage2_folder("news")
filings_df = load_stage2_folder("filings")

print(f"Broker: {len(broker_df)} stocks")
print(f"Foreign: {len(foreign_df)} stocks")
print(f"News: {len(news_df)} stocks")
print(f"Filings: {len(filings_df)} stocks")

Broker: 63 stocks
Foreign: 63 stocks
News: 63 stocks
Filings: 63 stocks


## Merge into the Main Leaderboard

Use `.merge(how="left")` the 63 shortlisted stocks will be fully populated, while the rest (344 stocks) will remain on the leaderboard but have `NaN` in their Stage 2 column. This is important: we are **not discarding** the unconfirmed stocks; we are simply marking them as not yet further validated.

In [3]:
final = leaderboard.copy()

if not broker_df.empty:
    final = final.merge(broker_df[["symbol", "broker_net_dominance", "top_accumulator", "top_distributor"]], on="symbol", how="left")
if not foreign_df.empty:
    final = final.merge(foreign_df[["symbol", "foreign_net_total"]], on="symbol", how="left")
if not news_df.empty:
    final = final.merge(news_df[["symbol", "n_news_articles"]], on="symbol", how="left")
if not filings_df.empty:
    final = final.merge(filings_df[["symbol", "n_filings"]], on="symbol", how="left")

print(f"Total rows: {len(final)}")
print(f"Stocks with complete Stage 2 data: {final['broker_net_dominance'].notna().sum()}")
final.head()

Total rows: 407
Stocks with complete Stage 2 data: 63


,symbol,date,close,open,high,low,volume,market_cap,return,return_zscore,...,vol_regime_ratio,price_anomaly_signal,volume_anomaly_signal,stage1_score,broker_net_dominance,top_accumulator,top_distributor,foreign_net_total,n_news_articles,n_filings
0,JECC,2026-09-10,825,690.0,825,680,957100,623700000000,0.250000,4.113743,...,1.222743,1.000000,1.00,1.000000,3.350000e+06,RB,OK,1.327875e+08,1.0,0.0
1,MGNA,2026-09-10,136,103.0,136,103,122855700,463912943560,0.346535,4.086236,...,1.065811,0.993313,1.00,0.996657,-6.564839e+08,MG,XA,-3.253452e+09,2.0,0.0
2,MSKY,2026-09-10,82,67.0,89,66,692576100,163538379360,0.242424,3.703918,...,1.242782,0.900377,1.00,0.950188,2.616402e+09,XL,CC,-1.905564e+09,1.0,0.0
3,VINS,2026-09-10,152,142.0,168,142,12044100,236121475368,0.070423,2.942694,...,0.738180,0.715333,1.00,0.857666,1.007110e+07,XL,XC,-2.700100e+07,0.0,0.0
4,KLIN,2026-09-10,127,115.0,127,115,2793300,166056351910,0.094828,2.846912,...,0.748228,0.692049,0.95,0.821025,1.022413e+08,OD,KK,-1.315077e+08,0.0,0.0


## Signal Calculation & Composite Score (Adaptive Weighting)

- `broker_signal`: the more negative the dominance (distribution > accumulation) → the higher the risk
- `foreign_signal`: the larger the foreign net outflow → the higher the risk
- `fundamental_signal`: binary – 1 if there is NO news or filing explaining the movement (no fundamental evidence), 0 if there is
- Weighting: stocks with complete Stage 2 data receive a combination of 4 signals; stocks without Stage 2 data receive a score based only on Stage 1, marked as `Unconfirmed`

In [ ]:
def minmax_norm(s: pd.Series) -> pd.Series:
    valid = s.dropna()
    if len(valid) < 2 or valid.max() == valid.min():
        return pd.Series(0.5, index=s.index)
    return (s - valid.min()) / (valid.max() - valid.min())

final["broker_signal"] = minmax_norm(-final["broker_net_dominance"]) if "broker_net_dominance" in final else np.nan
final["foreign_signal"] = minmax_norm(-final["foreign_net_total"]) if "foreign_net_total" in final else np.nan

if "n_news_articles" in final and "n_filings" in final:
    has_explanation = (final["n_news_articles"].fillna(0) > 0) | (final["n_filings"].fillna(0) > 0)
    final["fundamental_signal"] = np.where(
        final["broker_net_dominance"].notna(),
        (~has_explanation).astype(float),
        np.nan,
    )
else:
    final["fundamental_signal"] = np.nan

def compute_combined(row: pd.Series) -> float:
    has_stage2 = pd.notna(row.get("broker_signal"))
    if has_stage2:
        return (
            0.40 * row["stage1_score"]
            + 0.25 * row["broker_signal"]
            + 0.20 * row["foreign_signal"]
            + 0.15 * row["fundamental_signal"]
        )
    return row["stage1_score"]

final["combined_score"] = final.apply(compute_combined, axis=1)
final["is_confirmed"] = final["broker_signal"].notna()
final["risk_tier"] = np.where(final["is_confirmed"], "NORMAL", "UNCONFIRMED")

final = final.sort_values("combined_score", ascending=False).reset_index(drop=True)
final.to_csv(os.path.join(CACHE_DIR, "pumprisk_final_score.csv"), index=False)

print(f"Confirmed stocks: {final['is_confirmed'].sum()} / {len(final)}")
print("(risk_tier will be overwritten by calibrated thresholds further below)")

print("Risk tier distribution (confirmed Stage 2 stocks only):")
print(final[final["is_confirmed"]]["risk_tier"].value_counts())

final[final["is_confirmed"]][[
    "symbol", "combined_score", "risk_tier", "stage1_score",
    "broker_signal", "foreign_signal", "fundamental_signal",
    "top_accumulator", "top_distributor", "n_news_articles", "n_filings",
]].head(20)

Confirmed stocks: 63 / 407
(risk_tier will be overwritten by calibrated thresholds further below)
Risk tier distribution (confirmed Stage 2 stocks only):
risk_tier
NORMAL    63
Name: count, dtype: int64


,symbol,combined_score,risk_tier,stage1_score,broker_signal,foreign_signal,fundamental_signal,top_accumulator,top_distributor,n_news_articles,n_filings
0,JELI,0.754433,NORMAL,0.761084,1.000000,1.000000,0.0,CC,YU,7.0,0.0
15,VINS,0.556168,NORMAL,0.857666,0.116037,0.170463,1.0,XL,XC,0.0,0.0
32,KLIN,0.541277,NORMAL,0.821025,0.113531,0.172422,1.0,OD,KK,0.0,0.0
42,SEMA,0.518840,NORMAL,0.766880,0.106936,0.176770,1.0,CC,FS,0.0,0.0
43,PLAN,0.518574,NORMAL,0.771339,0.115897,0.155320,1.0,CP,XL,0.0,0.0
46,BAPI,0.515772,NORMAL,0.670512,0.208474,0.227243,1.0,XL,CP,0.0,0.0
47,LPLI,0.515559,NORMAL,0.756407,0.116733,0.169066,1.0,DH,IU,0.0,0.0
50,AYLS,0.512425,NORMAL,0.726821,0.133620,0.191459,1.0,LG,AI,0.0,0.0
52,NASA,0.511749,NORMAL,0.747355,0.114897,0.170412,1.0,XL,DH,0.0,0.0
54,MANG,0.511453,NORMAL,0.727247,0.137115,0.181377,1.0,XL,AO,0.0,0.0


## Backtesting against Stock Suspensions

This is the most crucial validation of the methodology regarding the "Technical Depth" criterion: checking whether stocks that were suspended by the IDX during our analysis period actually had high `combined_score` or `stage1_score` values. If so, that proves our model is capturing truly meaningful signals, not just noise.

In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY = os.environ.get("SECTORS_API_KEY", "")
BASE_URL = "https://api.sectors.app/v2"
HEADERS = {"Authorization": API_KEY}
CACHE_DIR = "cache"

SUSPENSIONS_CACHE = os.path.join(CACHE_DIR, "suspensions_raw.csv")
BT_START = (datetime.now() - timedelta(days=180)).strftime("%Y-%m-%d")
BT_END = datetime.now().strftime("%Y-%m-%d")

if os.path.exists(SUSPENSIONS_CACHE):
    print(f"Loading suspension data from local cache: {SUSPENSIONS_CACHE}")
    susp_df = pd.read_csv(SUSPENSIONS_CACHE)
    susp_df["suspension_date"] = pd.to_datetime(susp_df["suspension_date"])
else:
    all_suspensions = []
    offset = 0
    print(f"Retrieving all IDX suspension data ({BT_START} to {BT_END})")
    
    while True:
        try:
            resp = requests.get(
                f"{BASE_URL}/suspensions/",
                headers=HEADERS,
                params={"start": BT_START, "end": BT_END, "limit": 20, "offset": offset},
                timeout=20,
            )
            if resp.status_code == 429:
                print("Rate limit 429 detected, 10-second cooldown")
                time.sleep(10)
                continue
                
            if resp.status_code != 200:
                print(f"Error {resp.status_code}: {resp.text}")
                break
                
            payload = resp.json()
            results = payload.get("results", [])
            all_suspensions.extend(results)
            
            pagination = payload.get("pagination", {})
            if not pagination.get("has_next"):
                break
            offset = pagination.get("next_offset", offset + 20)
            time.sleep(0.3)
            
        except requests.exceptions.RequestException as e:
            print(f"Connection lost: {e}")
            break

    if not all_suspensions:
        raise ValueError("Suspension data is empty")

    susp_df = pd.DataFrame(all_suspensions)
    susp_df["symbol"] = susp_df["symbol"].str.replace(".JK", "", regex=False)
    susp_df["suspension_date"] = pd.to_datetime(susp_df["suspension_date"])
    susp_df.to_csv(SUSPENSIONS_CACHE, index=False)

print(f"Total IDX suspension cases retrieved: {len(susp_df)} cases")
susp_df.head(5)

Loading suspension data from local cache: cache\suspensions_raw.csv
Total IDX suspension cases retrieved: 60 cases


,symbol,suspension_date,reason,pdf_url
0,JARR,2026-09-11,Terjadinya peningkatan harga kumulatif yang si...,https://www.idx.co.id/Portals/0/StaticData/New...
1,GRPH,2026-09-10,Terjadinya peningkatan harga kumulatif yang si...,https://www.idx.co.id/Portals/0/StaticData/New...
2,LCKM,2026-09-10,Sehubungan dengan adanya ketidakpastian atas k...,https://www.idx.co.id/StaticData/NewsAndAnnoun...
3,MGLV,2026-09-09,Terjadinya peningkatan harga kumulatif yang si...,https://www.idx.co.id/Portals/0/StaticData/New...
4,UANG,2026-09-09,Terjadinya peningkatan harga kumulatif yang si...,https://www.idx.co.id/Portals/0/StaticData/New...


In [6]:
raw = pd.read_csv(os.path.join(CACHE_DIR, "daily_all_combined.csv"), parse_dates=["date"])
raw = raw.sort_values(["symbol", "date"])

ROLLING_WINDOW = 20

def compute_point_in_time(group, window=ROLLING_WINDOW):
    g = group.sort_values("date").copy()
    g["return"] = g["close"].pct_change()
    roll_mean = g["return"].rolling(window, min_periods=10).mean()
    roll_std = g["return"].rolling(window, min_periods=10).std()
    g["return_zscore"] = (g["return"] - roll_mean) / (roll_std + 1e-8)
    g["volume_percentile"] = g["volume"].rolling(window, min_periods=10).apply(
        lambda x: x.rank(pct=True).iloc[-1] if len(x) > 0 else np.nan, raw=False
    )
    return g

print("Calculating historical rolling features (Point-in-Time)")
raw_feat = raw.groupby("symbol", group_keys=False).apply(compute_point_in_time)

def minmax_norm(s):
    valid = s.dropna()
    if len(valid) < 2 or valid.max() == valid.min():
        return pd.Series(0.5, index=s.index)
    return (s - valid.min()) / (valid.max() - valid.min())

raw_feat["price_anomaly_signal"] = minmax_norm(raw_feat["return_zscore"].abs())
raw_feat["volume_anomaly_signal"] = raw_feat["volume_percentile"].fillna(0.5)
raw_feat["point_in_time_score"] = (
    0.5 * raw_feat["price_anomaly_signal"] + 0.5 * raw_feat["volume_anomaly_signal"]
)

matched = []
for _, row in susp_df.iterrows():
    sym, susp_date = row["symbol"], row["suspension_date"]
    hist = raw_feat[(raw_feat["symbol"] == sym) & (raw_feat["date"] < susp_date)]
    if hist.empty:
        continue
    last_row = hist.sort_values("date").iloc[-1]
    matched.append({
        "symbol": sym,
        "suspension_date": susp_date.date(),
        "last_trading_day_before": last_row["date"].date(),
        "score_before_suspension": last_row["point_in_time_score"],
        "return_zscore_before": last_row["return_zscore"],
        "reason": row.get("reason", "")
    })

matched_df = pd.DataFrame(matched).sort_values("score_before_suspension", ascending=False)
matched_df.to_csv(os.path.join(CACHE_DIR, "backtest_point_in_time.csv"), index=False)

Calculating historical rolling features (Point-in-Time)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21772\2851883882.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  raw_feat = raw.groupby("symbol", group_keys=False).apply(compute_point_in_time)


In [7]:
print(f"Suspension cases with data available in our history: {len(matched_df)} / {len(susp_df)}")

baseline_median = raw_feat["point_in_time_score"].median()
suspended_median = matched_df["score_before_suspension"].median()

print(f"Median score for all stocks (normal daily conditions): {baseline_median:.3f}")
print(f"Median score for stocks RIGHT BEFORE IDX suspension: {suspended_median:.3f}")

if suspended_median > baseline_median:
    print("VALIDATION SUCCESSFUL")
    print("Our model proves that stocks suspended by the IDX indeed exhibit significantly higher anomaly scores just before the suspension")

matched_df.head(15)

Suspension cases with data available in our history: 31 / 60
Median score for all stocks (normal daily conditions): 0.271
Median score for stocks RIGHT BEFORE IDX suspension: 0.552
VALIDATION SUCCESSFUL
Our model proves that stocks suspended by the IDX indeed exhibit significantly higher anomaly scores just before the suspension


,symbol,suspension_date,last_trading_day_before,score_before_suspension,return_zscore_before,reason
6,TMPO,2026-08-27,2026-08-26,0.740808,2.046161,Terjadinya peningkatan harga kumulatif yang si...
21,BAIK,2026-04-24,2026-04-23,0.734585,1.993284,Terjadinya peningkatan harga kumulatif yang si...
22,BAPA,2026-04-21,2026-04-20,0.722962,1.894517,Terjadinya peningkatan harga kumulatif yang si...
11,BAJA,2026-08-04,2026-08-03,0.702499,1.933076,Terjadinya peningkatan harga kumulatif yang si...
4,SAFE,2026-09-02,2026-09-01,0.681950,2.183323,Terjadinya peningkatan harga kumulatif yang si...
19,BOBA,2026-04-29,2026-04-28,0.673293,1.472476,Terjadinya peningkatan harga kumulatif yang si...
7,YPAS,2026-08-19,2026-08-18,0.615213,0.978972,Terjadinya peningkatan harga kumulatif yang si...
10,TRUK,2026-08-11,2026-08-10,0.596491,1.032319,Terjadinya peningkatan harga kumulatif yang si...
12,RGAS,2026-07-30,2026-07-29,0.588431,1.601113,Terjadinya peningkatan harga kumulatif yang si...
20,KING,2026-04-28,2026-04-27,0.583662,0.923304,Terjadinya peningkatan harga kumulatif yang si...


## Calibrate Risk Tier Thresholds From Empirical Distributions

Instead of guessing thresholds, we derive them from the two distributions we now have real data for: scores on normal trading days, and scores right before an actual IDX suspension.

The three candidate percentiles are not guaranteed to come out in the right order (in practice, P25 of the suspended-stock distribution came out *lower* than P90 of the normal-day distribution). Sorting them ascending before assigning HIGH/MEDIUM/WATCH guarantees HIGH > MEDIUM > WATCH regardless of which raw percentile came from which distribution.

### Threshold & Risk Tier Calibration

In [9]:
suspended_scores = matched_df["score_before_suspension"].dropna()
baseline_scores = raw_feat["point_in_time_score"].dropna()

high_candidate = np.percentile(suspended_scores, 25)
medium_candidate = np.percentile(baseline_scores, 90)
watch_candidate = np.percentile(baseline_scores, 75)

watch_threshold, medium_threshold, high_threshold = sorted(
    [high_candidate, medium_candidate, watch_candidate]
)

print("Calibrated Threshold (From Empirical Data)")
print(f"HIGH RISK >= {high_threshold:.3f}")
print(f"MEDIUM RISK >= {medium_threshold:.3f}")
print(f"WATCH >= {watch_threshold:.3f}")
print(f"NORMAL < {watch_threshold:.3f}")

caught_high = (suspended_scores >= high_threshold).mean()
caught_medium_or_higher = (suspended_scores >= medium_threshold).mean()
caught_watch_or_higher = (suspended_scores >= watch_threshold).mean()
print(f"\nRecall check against {len(suspended_scores)} actual IDX suspension cases:")
print(f"  {caught_high:.1%} flagged HIGH RISK")
print(f"  {caught_medium_or_higher:.1%} flagged at least MEDIUM RISK")
print(f"  {caught_watch_or_higher:.1%} flagged at least WATCH")

def risk_tier_calibrated(row: pd.Series) -> str:
    if not row["is_confirmed"]:
        return "UNCONFIRMED"
    s = row["combined_score"]
    if s >= high_threshold:
        return "HIGH RISK"
    elif s >= medium_threshold:
        return "MEDIUM RISK"
    elif s >= watch_threshold:
        return "WATCH"
    return "NORMAL"

final["risk_tier"] = final.apply(risk_tier_calibrated, axis=1)

pd.DataFrame([{
    "high_risk_threshold": high_threshold,
    "medium_risk_threshold": medium_threshold,
    "watch_threshold": watch_threshold,
    "note": "Thresholds sorted ascending from 3 candidates (P25 of suspended-stock scores, P90 & P75 of normal-day scores) to guarantee HIGH > MEDIUM > WATCH",
    "calibration_source": f"{len(susp_df)} historical IDX suspension cases ({BT_START} to {BT_END})",
    "n_suspension_raw": len(susp_df),
    "n_suspension_matched_universe": len(matched_df),
    "n_suspension_cases_used": len(suspended_scores),
    "baseline_median_score": float(baseline_scores.median()),
    "suspended_median_score": float(suspended_scores.median()),
    "recall_watch_or_higher": float(caught_watch_or_higher),
    "recall_medium_or_higher": float(caught_medium_or_higher),
    "recall_high": float(caught_high),
}]).to_csv(os.path.join(CACHE_DIR, "risk_tier_thresholds.csv"), index=False)

print("\nRisk tier distribution (confirmed Stage 2 stocks only):")
print(final[final["is_confirmed"]]["risk_tier"].value_counts())


Calibrated Threshold (From Empirical Data)
HIGH RISK >= 0.578
MEDIUM RISK >= 0.449
WATCH >= 0.309
NORMAL < 0.309

Recall check against 27 actual IDX suspension cases:
  37.0% flagged HIGH RISK
  63.0% flagged at least MEDIUM RISK
  74.1% flagged at least WATCH

Risk tier distribution (confirmed Stage 2 stocks only):
risk_tier
MEDIUM RISK    29
WATCH          25
NORMAL          8
HIGH RISK       1
Name: count, dtype: int64


### Calculate Normal Percentile & Save Final

In [10]:
final["percentile_vs_normal"] = final["stage1_score"].apply(
    lambda v: float((baseline_scores < v).mean() * 100) if pd.notna(v) else None
)

if os.path.exists(os.path.join(CACHE_DIR, "universe_sectors.csv")):
    sectors_df = pd.read_csv(os.path.join(CACHE_DIR, "universe_sectors.csv"))
    if "sub_sector" not in final.columns:
        final = final.merge(sectors_df, on="symbol", how="left")

final.to_csv(os.path.join(CACHE_DIR, "pumprisk_final_score.csv"), index=False)

print("percentile_vs_normal added to pumprisk_final_score.csv")
final[["symbol", "stage1_score", "percentile_vs_normal", "risk_tier"]].head(10)

percentile_vs_normal added to pumprisk_final_score.csv


,symbol,stage1_score,percentile_vs_normal,risk_tier
0,JELI,0.761084,98.141467,HIGH RISK
1,VAST,0.586444,90.579651,UNCONFIRMED
2,CLPI,0.583883,90.405351,UNCONFIRMED
3,PSDN,0.582276,90.273612,UNCONFIRMED
4,MIRA,0.581065,90.170247,UNCONFIRMED
5,ASHA,0.580754,90.156060,UNCONFIRMED
6,NANO,0.576172,89.809485,UNCONFIRMED
7,SHID,0.576072,89.793271,UNCONFIRMED
8,TARA,0.568418,89.134576,UNCONFIRMED
9,PTIS,0.565824,88.911634,UNCONFIRMED
